In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Time-variant Properties (e.g. coupon schedule) in LUSID 

Illustrates the use of time-variant properties, a type of property that depend on different effective dates.

Attributes
----------
coupon schedules
multi-valued properties
time-variant properties
"""

toggle_code("Toggle Docstring")

## Time-variant Properties

This notebook illustrates the use of time-variant properties, which are a type of property that depend on different effective dates. 

In the example below we use a quarterly ratings schedule as a demonstrative example, showing how the LUSID API can be used to query values on different effective dates. We will also demonstrate the [**bi-temporality**](https://support.finbourne.com/what-is-bi-temporal-data) of the data, using different [**'asAt'**](https://support.finbourne.com/what-is-asat-time) dates.  

In [ ]:
# Import lusid specific packages
# These are the core lusid packages for interacting with the API via Python

import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_portfolios_response,
    format_transactions_response,
    format_quotes_response,
)
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame

# Import libraries
from datetime import datetime, timedelta
import time
import pytz
import json
import os
import pandas as pd

# Set pandas dataframe display formatting
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

# Configure notebook logging and warnings
import logging
logging.basicConfig(level=logging.INFO)

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")

# Import required LUSID APIs
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)
instruments_api = api_factory.build(lu.InstrumentsApi)

print('LUSID Environment Initialised')
print('LUSID version : ', api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().build_version)

## 1. Load Data

### 1.1 Instruments

Load the instruments data from the source file and upload them into LUSID. The dataset in the example containins large cap stocks from the FTSE 100 Index, which we can use as examples to later setup our properties to. 

In [ ]:
# Read the instruments data
df = pd.read_csv("data/equity_transactions_isin.csv").drop_duplicates()
df.head()

In [ ]:
# Create a mapping schema for the instruments in the portfolio
instrument_mapping = {
    "identifier_mapping": {
        "ClientInternal": "ISIN"
    },
    "required": {
        "name": "name"
    },
}

# Instruments can be loaded using a dataframe with file_type set to "instruments"
result = load_from_data_frame(
    api_factory=api_factory,
    scope="TimeVariant",
    data_frame=df,
    mapping_required=instrument_mapping["required"],
    mapping_optional={},
    file_type="instruments",
    identifier_mapping=instrument_mapping["identifier_mapping"],
)

succ, failed, errors = format_instruments_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

### 1.2 Create the time-variant properties

With the instruments in LUSID, we can now define the properties and add them to the selected instruments. To this end we will need to define a [**property definition**](https://support.finbourne.com/what-is-a-property-definition), where we also need to specify the `life time` of the property as `TimeVariant`.  

In [ ]:
# Setup the property details
property_scope = "TimeVariantProperty"
property_code = "QuarterlyRating"

def create_property(property_scope, property_code):
    # Create the property definition request
    property_definition = models.CreatePropertyDefinitionRequest(
                domain="Instrument",
                scope=property_scope,
                code=property_code,
                display_name="Quarterly Ratings Estimates",
                data_type_id=models.ResourceId(scope="system", code="number"),
                life_time="TimeVariant",
            )

    # create property definition
    try:
        property_definition_request = property_definitions_api.create_property_definition(
            create_property_definition_request=property_definition
        )

    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property {property_definition.domain}/{property_definition.scope}/{property_definition.code} already exists"
            )
    return property_definition

# Pass our property scope and code to the property_definition
property_definition = create_property(property_scope, property_code)

### 1.3 Upsert the instrument properties

In order to upsert the instrument property, we create a function defining the body of our property request and use the `InstrumentsApi` to pass the values for our instrument.For more details see [**upsert_instruments_properties**](https://www.lusid.com/docs/api/#operation/UpsertInstrumentsProperties).

In this case we've used Aviva as an example, to which we are passing dates and numeric values to using the example schedule found below. The dates will be stored in the property's `effective_from` parameter as seen below.

In [ ]:
# set the property key using the property_definition -- this will follow the format domain/scope/code
property_key = f"{property_definition.domain}/{property_definition.scope}/{property_definition.code}"

# create a function to upsert the properties for a selected instrument using instruments_api
def upsert_instrument_property(ISIN, value, property_key, effectiveFrom):
    property_request = [
        models.UpsertInstrumentPropertyRequest(
            identifier_type="ClientInternal",
            identifier=ISIN,
            properties=[
                models.ModelProperty(
                    key=property_key,
                    value=models.PropertyValue(
                        metric_value=models.MetricValue(
                            value=value
                        )
                    ),
                )
            ]
        )
    ]

    response = instruments_api.upsert_instruments_properties(
                        upsert_instrument_property_request=property_request)
    print(f'Upserted property for ISIN: {instrument_id}, with Value: {element[key]} and Eff. Date: {date}')
    return response

# pass in the schedule effective dates and values for a selected ISIN
instrument_id = "GB0002162385"
schedule = [
    { "2020-12-31" : "5"},
    { "2021-03-31" : "4"},
    { "2021-06-30" : "3"},
    { "2021-09-30" : "3"},
]


for element in schedule:
    for key in element:
        date = datetime.strptime(key, "%Y-%m-%d").astimezone(pytz.utc)
        upsert_instrument_property(instrument_id, float(element[key]), property_key, effectiveFrom=date)

## 2. Querying Instrument Properties

### 2.1 Get properties by effective dates

We can use the [**get_instruments**](https://www.lusid.com/docs/api#operation/GetInstruments) call to the API along with the instrument property key in order to query the properties attached to the instrument for a given effective date. 

In [ ]:
# Create a function to query properties for a given instrument_id and effective date
def get_properties(ISIN, property_key, effective_date):
    response = instruments_api.get_instruments(
        identifier_type="ClientInternal",
        request_body=[ISIN],
        property_keys=[property_key],
        effective_at=datetime.strptime(effective_date, "%Y-%m-%d").astimezone(pytz.utc).isoformat(),
    )
    eff_from = response.values[ISIN].properties[0].effective_from
    if isinstance(eff_from, datetime):
        eff_from = eff_from.strftime('%m/%d/%Y %I:%M:%S %p') if eff_from.year > 1 else "Beginning of time"
    return (eff_from,
            response.values[ISIN].properties[0].value.metric_value.value)

dates=["2020-12-31",
       "2021-03-31",
       "2021-06-30",
       "2021-09-30"]

data = [get_properties(instrument_id, property_key, date) for date in dates]
df = pd.DataFrame(data, columns=['Effective Date', 'Value'])
display(df)

### 2.2 Viewing properties by different  _'as at'_ dates

The below is an example of the bi-temporality of data stored in LUSID, which means that we can view data as it was on a certain 'as at' date. In this case, we will assume that there was an update to the entries with our previous ratings example, requiring an amendment to the property values. 

Given the bi-temporal feature, we will still be able to view the original entries as they were before this amendment was made. In this example, we update the expected ratings using a hypothetical downgrade.  

In [ ]:
# Create a function to query properties for a given instrument_id and effective date
def get_properties_as_at(ISIN, effective_date, as_at_date):
    response = instruments_api.get_instruments(
        identifier_type="ClientInternal",
        request_body=[ISIN],
        property_keys=[property_key],
        effective_at=datetime.strptime(key, "%Y-%m-%d").astimezone(pytz.utc).isoformat(),
        as_at = as_at_date
    )
    # return the effective date and property value
    return (response.values[ISIN].properties[0].effective_from,
            as_at_date,
            response.values[ISIN].properties[0].value.metric_value.value)

In [ ]:
# Helper to safely format datetime for display
def safe_format_date(dt):
    if isinstance(dt, datetime):
        return dt.strftime('%m/%d/%Y %I:%M:%S %p') if dt.year > 1 else "Beginning of time"
    return str(dt)

# We begin by storing the current asAt value for '2021-06-30' before the change is made
time_now = datetime.now().astimezone(pytz.utc) 
data_1 = [get_properties_as_at(instrument_id, "2021-06-30", time_now)]
df_1 = pd.DataFrame(data_1, columns=['Effective Date', 'As at Date','Value'])
df_1['Effective Date'] = df_1['Effective Date'].apply(safe_format_date)
df_1['As at Date'] = df_1['As at Date'].apply(safe_format_date)


# Upsert the new set of data for the same instrument used before
new_schedule = [
    { "2020-12-31" : "5"},
    { "2021-03-31" : "2"},
    { "2021-06-30" : "1"},
    { "2021-09-30" : "1"},
]

for element in new_schedule:
    for key in element:
        date = datetime.strptime(key, "%Y-%m-%d").astimezone(pytz.utc)
        upsert_instrument_property(instrument_id, float(element[key]), property_key, effectiveFrom=date)

The [**get_instruments**](https://www.lusid.com/docs/api#operation/GetInstruments) call from the instruments API can be queried with the additional _'as at'_ parameter, which we will add to the get properties function below.  

In [ ]:
# Set the new as_at time to illustrate the updated value for the same effective_date
time_now = datetime.now().astimezone(pytz.utc) 

# Create a dataframe with the new updated time
data_2 = [get_properties_as_at(instrument_id, "2021-06-30", time_now)]
df_2 = pd.DataFrame(data_2, columns=['Effective Date', 'As at Date','Value'])
df_2['Effective Date'] = df_2['Effective Date'].apply(safe_format_date)
df_2['As at Date'] = df_2['As at Date'].apply(safe_format_date)


In [ ]:
# Show value for different as_at times
display(df_1)
display(df_2)